### Setup

In [ ]:
#from pygleif import GleifClient
import pandas as pd
import numpy as np
import sys
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

np.set_printoptions(threshold=sys.maxsize)


hmda_lei_fname = r"C:\Users\brade\Documents\Grad School\Capstone\HMDA_Join_LEI.csv"
outfname = "Legal_Entity_Summary.csv"

test_hmda_lei_fname = r"C:\Users\brade\Documents\Grad School\Capstone\Test_HMDA_Join_LEI.csv"
test_outfname = "Legal_Entity_Summary (Test IA Data).csv"

#hmda_lei_fname , outfname = test_hmda_lei_fname, test_outfname #Uncomment to test


In [23]:
hmda_lei = pd.read_csv(hmda_lei_fname)
hmda_lei = hmda_lei.fillna(0)   #Treat all nan values as 0, so that np.where can compare floats for all numerical (non coded) fields, 
                                # and so that nan can be used to replace, then remove exempt loans.  Note, coded colums (used for amortization flag) can only be 1, 2 or 1111
hmda_lei = hmda_lei[hmda_lei["action_taken"] == 1] #Filter to only originated loans
hmda_lei = hmda_lei.replace("Exempt",np.nan) #Replace fields with the value "Exempt" with nan
hmda_lei = hmda_lei.replace(1111,np.nan) #Replace fields with the value 1111 (code for exempt for certain fields) with nan
hmda_lei = hmda_lei[["activity_year","lei","Legal Name","action_taken","rate_spread","intro_rate_period","origination_charges","loan_amount","discount_points","lender_credits","negative_amortization","interest_only_payment","balloon_payment","other_nonamortizing_features","prepayment_penalty_term"]]
hmda_lei.dropna(inplace=True) #Remove all rows for loans that are exempt from reporting on fields used in this analysis

hmda_lei["Legal Name-LEI"] = hmda_lei["Legal Name"].astype(str) + "-" + hmda_lei["lei"].astype(str)

#Convert numeric columns from strings to floats
hmda_lei["rate_spread"] = hmda_lei["rate_spread"].astype(float)
hmda_lei["intro_rate_period"] = hmda_lei["intro_rate_period"].astype(float)
hmda_lei["origination_charges"] = hmda_lei["origination_charges"].astype(float)
hmda_lei["loan_amount"] = hmda_lei["loan_amount"].astype(float)
hmda_lei["origination_charges_percentage_loan_amount"] = hmda_lei["origination_charges"] / hmda_lei["loan_amount"]
hmda_lei["discount_points"] = hmda_lei["discount_points"].astype(float)
hmda_lei["lender_credits"] = hmda_lei["lender_credits"].astype(float)
hmda_lei["prepayment_penalty_term"] = hmda_lei["prepayment_penalty_term"].astype(float)

print(len(hmda_lei))
hmda_lei.head()

67117


,activity_year,lei,Legal Name,action_taken,rate_spread,intro_rate_period,origination_charges,loan_amount,discount_points,lender_credits,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,prepayment_penalty_term,Legal Name-LEI,origination_charges_percentage_loan_amount
9,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.000,0.0,0.00,155000.0,0.0,0.0,2.0,2.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",0.000000
11,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.296,1.0,0.00,745000.0,0.0,0.0,2.0,1.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",0.000000
44,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.200,0.0,772.01,105000.0,0.0,655.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,0.007352
46,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.564,0.0,0.00,395000.0,0.0,1837.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,0.000000
48,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,-0.586,0.0,2484.48,315000.0,0.0,650.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,0.007887


### Loan Level Predatory Behavior Flags

In [24]:
#print(sorted(hmda_lei["rate_spread"].unique()))
#hmda_lei["rate_spread"].unique()

In [25]:
#Interest_Rate_Flag
hmda_lei["Rate_Spread_Flag"] = np.where((hmda_lei["rate_spread"] > 2),True,False)

print(hmda_lei[hmda_lei["rate_spread"] > 2]["Rate_Spread_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["rate_spread"] <= 2]["Rate_Spread_Flag"].unique() == False) #Should be False Only

#All tests above should return [ True]

print(min(hmda_lei[hmda_lei["Rate_Spread_Flag"]==False]["rate_spread"].unique()))
print(max(hmda_lei[hmda_lei["Rate_Spread_Flag"]==False]["rate_spread"].unique()))
print(min(hmda_lei[hmda_lei["Rate_Spread_Flag"]==True]["rate_spread"].unique()))
print(max(hmda_lei[hmda_lei["Rate_Spread_Flag"]==True]["rate_spread"].unique()))

#hmda_lei[hmda_lei["Rate_Spread_Flag"]==False].head(10)
#hmda_lei[hmda_lei["Rate_Spread_Flag"]==True].head(10)


[ True]
[ True]
-825.0
2.0
2.001
178.0


In [26]:
#hmda_lei["intro_rate_period"].unique()

In [27]:
#Adjustable_Rate_Flag
hmda_lei["Adjustable_Rate_Flag"] = np.where(hmda_lei["intro_rate_period"] > 0 ,True,False)

#hmda_lei.head()

#Non-Predatory Values
print(hmda_lei[hmda_lei["intro_rate_period"] == 0]["Adjustable_Rate_Flag"].unique() == False) #Should be False Only
 
#Predatory Values
print(hmda_lei[hmda_lei["intro_rate_period"] == 1]["Adjustable_Rate_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["intro_rate_period"] == 62]["Adjustable_Rate_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["intro_rate_period"] == 84]["Adjustable_Rate_Flag"].unique() == True) #Should be True Only



#All tests above should return [ True]

print(min(hmda_lei[hmda_lei["Adjustable_Rate_Flag"]==False]["intro_rate_period"].unique()))
print(max(hmda_lei[hmda_lei["Adjustable_Rate_Flag"]==False]["intro_rate_period"].unique()))
print(min(hmda_lei[hmda_lei["Adjustable_Rate_Flag"]==True]["intro_rate_period"].unique()))
print(max(hmda_lei[hmda_lei["Adjustable_Rate_Flag"]==True]["intro_rate_period"].unique()))

hmda_lei[hmda_lei["Adjustable_Rate_Flag"] == True].head()


[ True]
[ True]
[ True]
[ True]
0.0
0.0
1.0
360.0


,activity_year,lei,Legal Name,action_taken,rate_spread,intro_rate_period,origination_charges,loan_amount,discount_points,lender_credits,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,prepayment_penalty_term,Legal Name-LEI,origination_charges_percentage_loan_amount,Rate_Spread_Flag,Adjustable_Rate_Flag
11,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.296,1.0,0.00,745000.0,0.00,0.0,2.0,1.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",0.000000,False,True
208,2025,549300HW662MN1WU8550,"UNITED WHOLESALE MORTGAGE, LLC",1,0.113,62.0,12549.98,445000.0,10454.98,25.0,2.0,2.0,2.0,2.0,0.0,"UNITED WHOLESALE MORTGAGE, LLC-549300HW662MN1W...",0.028202,False,True
345,2025,549300HW662MN1WU8550,"UNITED WHOLESALE MORTGAGE, LLC",1,0.953,84.0,6185.00,815000.0,4095.00,190.1,2.0,2.0,2.0,2.0,0.0,"UNITED WHOLESALE MORTGAGE, LLC-549300HW662MN1W...",0.007589,False,True
359,2025,549300HW662MN1WU8550,"UNITED WHOLESALE MORTGAGE, LLC",1,0.000,64.0,17682.34,375000.0,7336.29,228.0,2.0,2.0,2.0,2.0,0.0,"UNITED WHOLESALE MORTGAGE, LLC-549300HW662MN1W...",0.047153,False,True
388,2025,549300HW662MN1WU8550,"UNITED WHOLESALE MORTGAGE, LLC",1,-0.108,40.0,13448.36,285000.0,11373.36,270.0,2.0,2.0,2.0,2.0,0.0,"UNITED WHOLESALE MORTGAGE, LLC-549300HW662MN1W...",0.047187,False,True


In [28]:
#total_points_and_fees	origination_charges	discount_points	lender_credits

#print(hmda_lei["total_points_and_fees"].unique())
#print(sum(hmda_lei["origination_charges"])/len(hmda_lei["origination_charges"]))
#print(hmda_lei["discount_points"].unique())
#print(hmda_lei["lender_credits"].unique())


In [29]:
#Depracated
#Helper _Total_Points_And_Fees_Flag
#hmda_lei["_Total_Points_And_Fees_Flag"] = np.where(hmda_lei["total_points_and_fees"].astype(float) >0 ,True,False)

#WRITE TESTS!!!
#print(hmda_lei[hmda_lei["total_points_and_fees"] == 1615]["_Total_Points_And_Fees_Flag"].unique() == True) #Should be True Only
#print(hmda_lei[hmda_lei["total_points_and_fees"] == 0]["_Total_Points_And_Fees_Flag"].unique() == False) #Should be False

#All tests above should return [ True]


#print(min(hmda_lei[hmda_lei["_Total_Points_And_Fees_Flag"]==False]["total_points_and_fees"].unique()))
#print(max(hmda_lei[hmda_lei["_Total_Points_And_Fees_Flag"]==False]["total_points_and_fees"].unique()))
#print(min(hmda_lei[hmda_lei["_Total_Points_And_Fees_Flag"]==True]["total_points_and_fees"].unique()))
#print(max(hmda_lei[hmda_lei["_Total_Points_And_Fees_Flag"]==True]["total_points_and_fees"].unique()))



In [30]:
#Helper _Origination_Charges_Percentage_Loan_Amount_Flag 
hmda_lei["_Origination_Charges_Percentage_Loan_Amount_Flag"] = np.where(hmda_lei["origination_charges_percentage_loan_amount"].astype(float) >0.05 ,True,False)

#WRITE TESTS!!!
print(hmda_lei[hmda_lei["origination_charges"] == 772.01]["_Origination_Charges_Percentage_Loan_Amount_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["origination_charges"] == 0]["_Origination_Charges_Percentage_Loan_Amount_Flag"].unique() == False) #Should be False

#All tests above should return [ True]


print(min(hmda_lei[hmda_lei["_Origination_Charges_Percentage_Loan_Amount_Flag"]==False]["origination_charges_percentage_loan_amount"]))
print(max(hmda_lei[hmda_lei["_Origination_Charges_Percentage_Loan_Amount_Flag"]==False]["origination_charges_percentage_loan_amount"]))
print(min(hmda_lei[hmda_lei["_Origination_Charges_Percentage_Loan_Amount_Flag"]==True]["origination_charges_percentage_loan_amount"]))
print(max(hmda_lei[hmda_lei["_Origination_Charges_Percentage_Loan_Amount_Flag"]==True]["origination_charges_percentage_loan_amount"]))



[False]
[ True]
0.0
0.05
0.050006
0.10103333333333334


In [31]:
#Helper _Discount_Points_Flag

hmda_lei["_Discount_Points_Flag"] = np.where(hmda_lei["discount_points"].astype(float) >0 ,True,False)


#WRITE TESTS!!!
print(hmda_lei[hmda_lei["discount_points"] == 14132.61]["_Discount_Points_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["discount_points"] == 0]["_Discount_Points_Flag"].unique() == False) #Should be False

#All tests above should return [ True]


print(min(hmda_lei[hmda_lei["_Discount_Points_Flag"]==False]["discount_points"].unique()))
print(max(hmda_lei[hmda_lei["_Discount_Points_Flag"]==False]["discount_points"].unique()))
print(min(hmda_lei[hmda_lei["_Discount_Points_Flag"]==True]["discount_points"].unique()))
print(max(hmda_lei[hmda_lei["_Discount_Points_Flag"]==True]["discount_points"].unique()))

[ True]
[ True]
0.0
0.0
0.01
36028.24


In [32]:
#Helper _Lender_Credits_Flag

hmda_lei["_Lender_Credits_Flag"] = np.where(hmda_lei["lender_credits"].astype(float) >0 ,True,False)


#WRITE TESTS!!!
print(hmda_lei[hmda_lei["lender_credits"] == 655]["_Lender_Credits_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["lender_credits"] == 0]["_Lender_Credits_Flag"].unique() == False) #Should be False

#All tests above should return [ True]


print(min(hmda_lei[hmda_lei["_Lender_Credits_Flag"]==False]["lender_credits"].unique()))
print(max(hmda_lei[hmda_lei["_Lender_Credits_Flag"]==False]["lender_credits"].unique()))
print(min(hmda_lei[hmda_lei["_Lender_Credits_Flag"]==True]["lender_credits"].unique()))
print(max(hmda_lei[hmda_lei["_Lender_Credits_Flag"]==True]["lender_credits"].unique()))

[ True]
[ True]
0.0
0.0
0.01
14833.0


In [ ]:
#Origination_Charge_Flag
hmda_lei["All_Origination_Charges_Points_Credits_Flag"] = np.where((hmda_lei["_Origination_Charges_Percentage_Loan_Amount_Flag"]) | 
                                                (hmda_lei["_Discount_Points_Flag"]) | (hmda_lei["_Lender_Credits_Flag"]),True,False)

hmda_lei["_Discount_Points_or_Lender_Credits_Flag"] = np.where((hmda_lei["_Discount_Points_Flag"]) | (hmda_lei["_Lender_Credits_Flag"]),True,False)

hmda_lei.head()


,activity_year,lei,Legal Name,action_taken,rate_spread,intro_rate_period,origination_charges,loan_amount,discount_points,lender_credits,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,prepayment_penalty_term,Legal Name-LEI,origination_charges_percentage_loan_amount,Rate_Spread_Flag,Adjustable_Rate_Flag,_Origination_Charges_Percentage_Loan_Amount_Flag,_Discount_Points_Flag,_Lender_Credits_Flag,All_Origination_Charges_Flag,_Discount_Points_or_Lender_Credits_Flag
9,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.000,0.0,0.00,155000.0,0.0,0.0,2.0,2.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",0.000000,False,False,False,False,False,False,False
11,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.296,1.0,0.00,745000.0,0.0,0.0,2.0,1.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",0.000000,False,True,False,False,False,False,False
44,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.200,0.0,772.01,105000.0,0.0,655.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,0.007352,False,False,False,False,True,True,True
46,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.564,0.0,0.00,395000.0,0.0,1837.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,0.000000,False,False,False,False,True,True,True
48,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,-0.586,0.0,2484.48,315000.0,0.0,650.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,0.007887,False,False,False,False,True,True,True


In [34]:
#negative_amortization	interest_only_payment	balloon_payment	other_nonamortizing_features
#For all, code 1=exhibits behavior, code 2=does not exhibit behavior, code 1111=exempt

print(hmda_lei["negative_amortization"].unique())
print(hmda_lei["interest_only_payment"].unique())
print(hmda_lei["balloon_payment"].unique())
print(hmda_lei["other_nonamortizing_features"].unique())

[2. 1.]
[2. 1.]
[2. 1.]
[2. 1.]


In [35]:
#Non_Amortizing_Flag
hmda_lei["Non_Amortizing_Flag"] = np.where((hmda_lei["negative_amortization"] == 1) | (hmda_lei["interest_only_payment"] == 1) | (hmda_lei["balloon_payment"] == 1) | (hmda_lei["other_nonamortizing_features"] == 1),True,False)

hmda_lei.head()

#Non Predatory Behaviors - Should probably implement tests with multiple conditions given the OR nature of this flag, could check that sum of 4 cols is 8 if false, 3<x<8 if true

#Predatory Behaviors
print(hmda_lei[hmda_lei["negative_amortization"]==1]["Non_Amortizing_Flag"].unique()) #Should be True Only
print(hmda_lei[hmda_lei["interest_only_payment"]==1]["Non_Amortizing_Flag"].unique()) #Should be True Only
print(hmda_lei[hmda_lei["balloon_payment"]==1]["Non_Amortizing_Flag"].unique()) #Should be True Only
print(hmda_lei[hmda_lei["other_nonamortizing_features"]==1]["Non_Amortizing_Flag"].unique()) #Should be True Only

hmda_lei[["negative_amortization","interest_only_payment","balloon_payment","other_nonamortizing_features","Non_Amortizing_Flag"]].head(30)

hmda_lei[["negative_amortization","interest_only_payment","balloon_payment","other_nonamortizing_features","Non_Amortizing_Flag"]].head(30)


[ True]
[ True]
[ True]
[ True]


,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,Non_Amortizing_Flag
9,2.0,2.0,2.0,2.0,False
11,2.0,1.0,2.0,2.0,True
44,2.0,2.0,2.0,2.0,False
46,2.0,2.0,2.0,2.0,False
48,2.0,2.0,2.0,2.0,False
51,2.0,2.0,2.0,2.0,False
52,2.0,2.0,2.0,2.0,False
55,2.0,2.0,2.0,2.0,False
61,2.0,2.0,2.0,2.0,False
63,2.0,2.0,2.0,2.0,False


In [36]:
hmda_lei["prepayment_penalty_term"].unique()
#hmda_lei[pd.isna(hmda_lei["prepayment_penalty_term"])]["prepayment_penalty_term"].unique()

array([ 0., 24.,  2., 36., 12., 60.])

In [37]:
#Prepayment_Penalty_Flag
hmda_lei["Prepayment_Penalty_Flag"] = np.where((hmda_lei["prepayment_penalty_term"] > 0 ),True,False)

#hmda_lei.head()
#hmda_lei["Prepayment_Penalty_Flag"].unique()

#Non-Predatory Values
print(hmda_lei[hmda_lei["prepayment_penalty_term"] == 0]["Prepayment_Penalty_Flag"].unique() == False) #Should be False Only

#Predatory Values
print(hmda_lei[hmda_lei["prepayment_penalty_term"] == 24]["Prepayment_Penalty_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["prepayment_penalty_term"] == 36]["Prepayment_Penalty_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["prepayment_penalty_term"] == 2]["Prepayment_Penalty_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["prepayment_penalty_term"] == 12]["Prepayment_Penalty_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["prepayment_penalty_term"] == 60]["Prepayment_Penalty_Flag"].unique() == True) #Should be True Only

#All tests above should return [ True]


print(min(hmda_lei[hmda_lei["Prepayment_Penalty_Flag"]==False]["prepayment_penalty_term"].unique()))
print(max(hmda_lei[hmda_lei["Prepayment_Penalty_Flag"]==False]["prepayment_penalty_term"].unique()))
print(min(hmda_lei[hmda_lei["Prepayment_Penalty_Flag"]==True]["prepayment_penalty_term"].unique()))
print(max(hmda_lei[hmda_lei["Prepayment_Penalty_Flag"]==True]["prepayment_penalty_term"].unique()))



[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
0.0
0.0
2.0
60.0


In [38]:
hmda_lei.head()

,activity_year,lei,Legal Name,action_taken,rate_spread,intro_rate_period,origination_charges,loan_amount,discount_points,lender_credits,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,prepayment_penalty_term,Legal Name-LEI,origination_charges_percentage_loan_amount,Rate_Spread_Flag,Adjustable_Rate_Flag,_Origination_Charges_Percentage_Loan_Amount_Flag,_Discount_Points_Flag,_Lender_Credits_Flag,All_Origination_Charges_Flag,_Discount_Points_or_Lender_Credits_Flag,Non_Amortizing_Flag,Prepayment_Penalty_Flag
9,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.000,0.0,0.00,155000.0,0.0,0.0,2.0,2.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",0.000000,False,False,False,False,False,False,False,False,False
11,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.296,1.0,0.00,745000.0,0.0,0.0,2.0,1.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",0.000000,False,True,False,False,False,False,False,True,False
44,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.200,0.0,772.01,105000.0,0.0,655.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,0.007352,False,False,False,False,True,True,True,False,False
46,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.564,0.0,0.00,395000.0,0.0,1837.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,0.000000,False,False,False,False,True,True,True,False,False
48,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,-0.586,0.0,2484.48,315000.0,0.0,650.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,0.007887,False,False,False,False,True,True,True,False,False


In [ ]:
#Loan-Level 
hmda_lei["Predatory_Loan(Any_Behavior)"] = np.where(hmda_lei["Rate_Spread_Flag"] | hmda_lei["Adjustable_Rate_Flag"]| hmda_lei["All_Origination_Charges_Points_Credits_Flag"]| hmda_lei["Non_Amortizing_Flag"]| hmda_lei["Prepayment_Penalty_Flag"],True,False)
#hmda_lei["Predatory_Loan(Any_Behavior_Except_Regular_Origination_Charges)"] = np.where(hmda_lei["Rate_Spread_Flag"] | hmda_lei["Adjustable_Rate_Flag"]| hmda_lei["_Discount_Points_or_Lender_Credits_Flag"]| hmda_lei["Non_Amortizing_Flag"]| hmda_lei["Prepayment_Penalty_Flag"],True,False)
hmda_lei["Loan_Count_Predatory_Behaviors"] = hmda_lei[["Rate_Spread_Flag","Adjustable_Rate_Flag","All_Origination_Charges_Flag","Non_Amortizing_Flag","Prepayment_Penalty_Flag"]].sum(axis=1)

#hmda_lei[hmda_lei["Rate_Spread_Flag"] == True].head()
hmda_lei.head()

,activity_year,lei,Legal Name,action_taken,rate_spread,intro_rate_period,origination_charges,loan_amount,discount_points,lender_credits,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,prepayment_penalty_term,Legal Name-LEI,origination_charges_percentage_loan_amount,Rate_Spread_Flag,Adjustable_Rate_Flag,_Origination_Charges_Percentage_Loan_Amount_Flag,_Discount_Points_Flag,_Lender_Credits_Flag,All_Origination_Charges_Flag,_Discount_Points_or_Lender_Credits_Flag,Non_Amortizing_Flag,Prepayment_Penalty_Flag,Predatory_Loan(Any_Behavior),Predatory_Loan(Any_Behavior_Except_Regular_Origination_Charges),Loan_Count_Predatory_Behaviors
9,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.000,0.0,0.00,155000.0,0.0,0.0,2.0,2.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",0.000000,False,False,False,False,False,False,False,False,False,False,False,0
11,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.296,1.0,0.00,745000.0,0.0,0.0,2.0,1.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",0.000000,False,True,False,False,False,False,False,True,False,True,True,2
44,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.200,0.0,772.01,105000.0,0.0,655.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,0.007352,False,False,False,False,True,True,True,False,False,True,True,1
46,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.564,0.0,0.00,395000.0,0.0,1837.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,0.000000,False,False,False,False,True,True,True,False,False,True,True,1
48,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,-0.586,0.0,2484.48,315000.0,0.0,650.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,0.007887,False,False,False,False,True,True,True,False,False,True,True,1


### Aggregation Functions

In [ ]:
print("Summary of Predatory Behaviors Across All Loans","\n---------------------------------------------------------------------------------------\n")
print("Share of all loans with a rate spread over 2 percentage points:", "{:.2%}".format(hmda_lei["Rate_Spread_Flag"].sum()/hmda_lei["lei"].count()),'\n')
print("Share of all loans with an adjustable rate mortgage:", "{:.2%}".format(hmda_lei["Adjustable_Rate_Flag"].sum()/hmda_lei["lei"].count()),'\n')

print("Share of all loans with origination charges (including discount points and/or lending credits):", "{:.2%}".format(hmda_lei["All_Origination_Charges_Flag"].sum()/hmda_lei["lei"].count()))
print("\tShare of all loans with discount points:", "{:.2%}".format(hmda_lei["_Discount_Points_Flag"].sum()/hmda_lei["lei"].count()))
print("\tShare of all loans with lender credits:", "{:.2%}".format(hmda_lei["_Lender_Credits_Flag"].sum()/hmda_lei["lei"].count()))
print("\tShare of all loans excessive origination charges (>5%):", "{:.2%}".format(hmda_lei["_Origination_Charges_Percentage_Loan_Amount_Flag"].sum()/hmda_lei["lei"].count()),'\n')
print("\tShare of all loans with discount points or lender credits (not including excessive origination charges):", "{:.2%}".format(hmda_lei["_Discount_Points_or_Lender_Credits_Flag"].sum()/hmda_lei["lei"].count()),'\n')

print("Share of all loans that are non-amortizing (including interest only or balloon payments):", "{:.2%}".format(hmda_lei["Non_Amortizing_Flag"].sum()/hmda_lei["lei"].count()))
print("\tShare of all loans with basic non-amortizing features:", "{:.2%}".format(hmda_lei[hmda_lei["negative_amortization"]==1]["negative_amortization"].sum()/hmda_lei["lei"].count()))
print("\tShare of all loans with interest-only payments:", "{:.2%}".format(hmda_lei[hmda_lei["interest_only_payment"]==1]["interest_only_payment"].sum()/hmda_lei["lei"].count()))
print("\tShare of all loans with balloon payments:", "{:.2%}".format(hmda_lei[hmda_lei["balloon_payment"]==1]["balloon_payment"].sum()/hmda_lei["lei"].count()))
print("\tShare of all loans with other non-amortizing features:", "{:.2%}".format(hmda_lei[hmda_lei["other_nonamortizing_features"]==1]["other_nonamortizing_features"].sum()/hmda_lei["lei"].count()),'\n')

print("Share of all loans with a prepayment penalty:", "{:.2%}".format(hmda_lei["Prepayment_Penalty_Flag"].sum()/hmda_lei["lei"].count()),'\n')

print("Share of all loans with any predatory behavior:", "{:.2%}".format(hmda_lei["Predatory_Loan(Any_Behavior)"].sum()/hmda_lei["lei"].count()))


Summary of Predatory Behaviors Across All Loans 
---------------------------------------------------------------------------------------

Share of all loans with a rate spread over 2 percentage points: 7.31% 

Share of all loans with an adjustable rate mortgage: 19.27% 

Share of all loans with origination charges (including discount points and/or lending credits): 36.98%
	Share of all loans with discount points: 20.07%
	Share of all loans with lender credits: 21.59%
	Share of all loans excessive origination charges (>5%): 0.54% 

	Share of all loans with discount points or lender credits (not including excessive origination charges): 36.95% 

Share of all loans that are non-amortizing (including interest only or balloon payments): 18.08%
	Share of all loans with basic non-amortizing features: 0.03%
	Share of all loans with interest-only payments: 7.74%
	Share of all loans with balloon payments: 14.92%
	Share of all loans with other non-amortizing features: 0.18% 

Share of all loans w

In [ ]:
Legal_Entity_Summary = hmda_lei.groupby("Legal Name-LEI").agg(
Total_Loans = ("lei","count"),
Total_High_Rate_Spread = ("Rate_Spread_Flag","sum"),
Total_Adjustable_Rate = ("Adjustable_Rate_Flag","sum"),
Total_Origination_Charges = ("All_Origination_Charges_Flag","sum"),
Total_Discount_Points_or_Lender_Credits = ("_Discount_Points_or_Lender_Credits_Flag","sum"),
Total_Non_Amortizing = ("Non_Amortizing_Flag","sum"),
Total_Prepayment_Penalty = ("Prepayment_Penalty_Flag","sum"),
Total_Predatory_Loan_Any_Behavior = ("Predatory_Loan(Any_Behavior)","sum"),
Total_Predatory_Loan_Any_Behavior_Except_Regular_Origination_Charges = ("Predatory_Loan(Any_Behavior_Except_Regular_Origination_Charges)","sum")
)

Legal_Entity_Summary["Share_High_Rate_Spread"] = Legal_Entity_Summary["Total_High_Rate_Spread"] / Legal_Entity_Summary["Total_Loans"]
Legal_Entity_Summary["Share_Adjustable_Rate"] = Legal_Entity_Summary["Total_Adjustable_Rate"] / Legal_Entity_Summary["Total_Loans"]
Legal_Entity_Summary["Share_Origination_Charges"] = Legal_Entity_Summary["Total_Origination_Charges"] / Legal_Entity_Summary["Total_Loans"]
Legal_Entity_Summary["Share__Discount_Points_or_Lender_Credits_Flag"] = Legal_Entity_Summary["Total_Discount_Points_or_Lender_Credits"] / Legal_Entity_Summary["Total_Loans"]
Legal_Entity_Summary["Share_Non_Amortizing"] = Legal_Entity_Summary["Total_Non_Amortizing"] / Legal_Entity_Summary["Total_Loans"]
Legal_Entity_Summary["Share_Prepayment_Penalty"] = Legal_Entity_Summary["Total_Prepayment_Penalty"] / Legal_Entity_Summary["Total_Loans"]
Legal_Entity_Summary["Share_Predatory_Loan(Any_Behavior)"] = Legal_Entity_Summary["Total_Predatory_Loan_Any_Behavior"] / Legal_Entity_Summary["Total_Loans"]
Legal_Entity_Summary["Share_Predatory_Loan(Any_Behavior_Except_Regular_Origination_Charges)"] = Legal_Entity_Summary["Total_Predatory_Loan_Any_Behavior_Except_Regular_Origination_Charges"] / Legal_Entity_Summary["Total_Loans"]



Legal_Entity_Summary.to_csv(outfname)
print(len(Legal_Entity_Summary[Legal_Entity_Summary["Total_Loans"] > 50]))
Legal_Entity_Summary[Legal_Entity_Summary["Total_Loans"] > 50].sort_values(by="Share_Predatory_Loan(Any_Behavior_Except_Regular_Origination_Charges)",ascending=False)



2430


,Total_Loans,Total_High_Rate_Spread,Total_Adjustable_Rate,Total_Origination_Charges,Total_Discount_Points_or_Lender_Credits,Total_Non_Amortizing,Total_Prepayment_Penalty,Total_Predatory_Loan_Any_Behavior,Total_Predatory_Loan_Any_Behavior_Except_Regular_Origination_Charges,Share_High_Rate_Spread,Share_Adjustable_Rate,Share_Origination_Charges,Share__Discount_Points_or_Lender_Credits_Flag,Share_Non_Amortizing,Share_Prepayment_Penalty,Share_Predatory_Loan(Any_Behavior),Share_Predatory_Loan(Any_Behavior_Except_Regular_Origination_Charges)
Legal Name-LEI,,,,,,,,,,,,,,,,,
Mid Oregon Federal Credit Union-549300TEGBOIDI6Z5J56,528,16,0,0,0,528,0,528,528,0.030303,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000
"NEXUS SERIES B, LLC-549300X0DCWAGU4DZH55",79,0,0,0,0,79,0,79,79,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000
"NORTHMARQ CAPITAL FINANCE, L.L.C.-549300AV8QD552DSI743",136,0,0,0,0,136,0,136,136,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000
BELLCO-5493008L9O5NV7EAK360,2822,1,2819,0,0,2822,0,2822,2822,0.000354,0.998937,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000
Polish-American Federal Credit Union-254900V7ZAKK6Z6SQ303,102,0,101,79,0,1,0,102,102,0.000000,0.990196,0.774510,0.000000,0.009804,0.000000,1.000000,1.000000
NEIGHBORHOOD-549300I0JGZE27J13Z50,168,9,0,168,168,0,0,168,168,0.053571,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000
BayFirst National Bank-549300DMI3W6YLDVSK93,396,21,390,0,0,396,376,396,396,0.053030,0.984848,0.000000,0.000000,1.000000,0.949495,1.000000,1.000000
Openland Credit Union-549300JX4XZ2XQAMU684,169,29,0,168,168,7,0,169,169,0.171598,0.000000,0.994083,0.994083,0.041420,0.000000,1.000000,1.000000
"PGIM REAL ESTATE FINANCE, LLC-B2S31CFVSWTN3FR00Q90",93,0,17,0,0,80,0,93,93,0.000000,0.182796,0.000000,0.000000,0.860215,0.000000,1.000000,1.000000
